# QC cutoff analysis

Proposes per-sample QC cutoffs from the data. Nothing is filtered or saved here; the current cutoffs
(`MIN_COUNTS = 15`, `MIN_GENES = 10` in `02_qc_filtering.ipynb`) are shown on every plot for comparison.

1. **Noise floor** from Xenium's negative-control estimate of false-positive transcripts per cell.
2. **Two-component Gaussian mixture** on log10 counts and log10 genes, cut where the low component stops being more likely.
3. **MAD outliers** and **cell area** bounds as sanity checks.
4. **Sweep** of candidate count cutoffs against how sparse the tissue is around the removed cells.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
from scipy.stats import norm, poisson, median_abs_deviation
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors

In [ ]:
SAMPLES_DIR  = '/home/workspace/PCD_data/xenium/commercial/data/samples'
METRICS_PATH = '/home/workspace/PCD_data/xenium/commercial/data/metrics_summary_all.csv'

SAMPLE_NAMES = [
    'TIS05393',
    'TIS04788',
    'TIS05392',
    'TIS05394',
    'TIS04789',
    'TIS05397_008',
]

CURRENT_MIN_COUNTS = 15
CURRENT_MIN_GENES  = 10

NOISE_QUANTILE = 0.999
NMADS          = 5

## Load

Reads the per-sample `.h5ad` files directly (read-only). Only a small QC table and the coordinates are kept per sample.

In [ ]:
qc      = {}
spatial = {}
for name in SAMPLE_NAMES:
    adata = ad.read_h5ad(f'{SAMPLES_DIR}/{name}.h5ad')
    cells = adata.obs[['total_counts', 'cell_area']].copy()
    # Adds n_genes (genes with at least one count) to this per-sample QC table in memory only; adata is not modified.
    cells['n_genes'] = np.asarray((adata.X > 0).sum(axis=1)).ravel()
    qc[name]      = cells
    spatial[name] = adata.obsm['spatial']

## 1. Noise floor

`estimated_number_of_false_positive_transcripts_per_cell` in the Xenium metrics is the expected number of
counts per cell from background. A cell whose counts are within the upper tail of a Poisson with that mean
cannot be told apart from noise.

In [ ]:
metrics = pd.read_csv(METRICS_PATH).set_index('sample')
noise_rate = metrics.loc[SAMPLE_NAMES, 'estimated_number_of_false_positive_transcripts_per_cell']
noise_min_counts = pd.Series(poisson.ppf(NOISE_QUANTILE, noise_rate) + 1, index=SAMPLE_NAMES)

pd.DataFrame({'false_positives_per_cell': noise_rate, 'noise_min_counts': noise_min_counts})

## 2. Gaussian mixture

Fitted on cells with at least one count (zero-count cells are removed by any cutoff and would add a spike at log10(0)).
`bic_gain` is BIC(one component) minus BIC(two components). With around 100k cells it is large almost whatever the shape, so it does not show that two real humps exist; judge that from the histograms below.

In [ ]:
mixtures     = {}
mixture_rows = []
for name in SAMPLE_NAMES:
    for metric in ['total_counts', 'n_genes']:
        values     = qc[name][metric].to_numpy()
        log_values = np.log10(values[values > 0]).reshape(-1, 1)

        one_component  = GaussianMixture(1, random_state=0).fit(log_values)
        two_components = GaussianMixture(2, random_state=0).fit(log_values)

        low  = np.argmin(two_components.means_.ravel())
        high = 1 - low
        grid = np.linspace(two_components.means_.min(), two_components.means_.max(), 2000).reshape(-1, 1)
        posterior_low = two_components.predict_proba(grid)[:, low]
        cut_log10     = grid[np.argmax(posterior_low < 0.5), 0]

        mixtures[(name, metric)] = two_components
        mixture_rows.append({
            'sample': name,
            'metric': metric,
            'fraction_low': two_components.weights_[low],
            'mean_low': 10 ** two_components.means_[low, 0],
            'mean_high': 10 ** two_components.means_[high, 0],
            'min_value': np.ceil(10 ** cut_log10),
            'bic_gain': one_component.bic(log_values) - two_components.bic(log_values),
        })

mixture_table = pd.DataFrame(mixture_rows)
mixture_table

## 3. MAD and area bounds

Bounds are median ± `NMADS` median absolute deviations on `log1p` of each metric. A lower bound at or below zero means the rule sets no lower cutoff for that sample.

In [ ]:
mad_rows = []
for name in SAMPLE_NAMES:
    row = {'sample': name}
    for metric in ['total_counts', 'n_genes', 'cell_area']:
        logged = np.log1p(qc[name][metric].to_numpy())
        center = np.median(logged)
        spread = NMADS * median_abs_deviation(logged)
        row[f'{metric}_low']  = np.expm1(center - spread)
        row[f'{metric}_high'] = np.expm1(center + spread)
    mad_rows.append(row)

mad_table = pd.DataFrame(mad_rows).set_index('sample').round(1)
mad_table

## Proposed cutoffs

Counts: the larger of the noise floor and the mixture cut. Genes: the mixture cut. Cells are kept when they are at or above both.

In [ ]:
mixture_wide = mixture_table.pivot(index='sample', columns='metric')

cutoffs = pd.DataFrame(index=SAMPLE_NAMES)
cutoffs['noise_min_counts']    = noise_min_counts
cutoffs['gmm_min_counts']      = mixture_wide[('min_value', 'total_counts')]
cutoffs['gmm_min_genes']       = mixture_wide[('min_value', 'n_genes')]
cutoffs['bic_gain_counts']     = mixture_wide[('bic_gain', 'total_counts')].round(0)
cutoffs['bic_gain_genes']      = mixture_wide[('bic_gain', 'n_genes')].round(0)
cutoffs['proposed_min_counts'] = cutoffs[['noise_min_counts', 'gmm_min_counts']].max(axis=1)
cutoffs['proposed_min_genes']  = cutoffs['gmm_min_genes']

for name in SAMPLE_NAMES:
    counts = qc[name]['total_counts']
    genes  = qc[name]['n_genes']
    current_removed  = (counts < CURRENT_MIN_COUNTS) | (genes < CURRENT_MIN_GENES)
    proposed_removed = (counts < cutoffs.loc[name, 'proposed_min_counts']) | (genes < cutoffs.loc[name, 'proposed_min_genes'])
    cutoffs.loc[name, 'median_counts']           = counts.median()
    cutoffs.loc[name, 'fraction_removed_current']  = current_removed.mean()
    cutoffs.loc[name, 'fraction_removed_proposed'] = proposed_removed.mean()

cutoffs.round(3)

## Cutoffs on the distributions

Grey is the histogram of log10 values (cells with at least one count). Coloured curves are the two fitted components.
Red dashed is the current cutoff, black is the mixture cut, green dotted is the noise floor.

In [ ]:
fig, axes = plt.subplots(len(SAMPLE_NAMES), 2, figsize=(14, 3.6 * len(SAMPLE_NAMES)), squeeze=False, gridspec_kw={'hspace': 0.5})
for row_axes, name in zip(axes, SAMPLE_NAMES):
    for ax, metric, current, proposed in zip(
        row_axes,
        ['total_counts', 'n_genes'],
        [CURRENT_MIN_COUNTS, CURRENT_MIN_GENES],
        [cutoffs.loc[name, 'gmm_min_counts'], cutoffs.loc[name, 'gmm_min_genes']],
    ):
        values     = qc[name][metric].to_numpy()
        log_values = np.log10(values[values > 0])

        ax.hist(log_values, bins=80, color='lightgray')
        bin_width = (log_values.max() - log_values.min()) / 80
        x = np.linspace(log_values.min(), log_values.max(), 400)
        mixture = mixtures[(name, metric)]
        for weight, mean, variance in zip(mixture.weights_, mixture.means_.ravel(), mixture.covariances_.ravel()):
            ax.plot(x, len(log_values) * bin_width * weight * norm.pdf(x, mean, np.sqrt(variance)), lw=1.5)

        ax.axvline(np.log10(current),  color='tab:red', ls='--', label=f'current ({current})')
        ax.axvline(np.log10(proposed), color='black',            label=f'mixture cut ({proposed:.0f})')
        if metric == 'total_counts':
            ax.axvline(np.log10(noise_min_counts[name]), color='tab:green', ls=':', label=f'noise floor ({noise_min_counts[name]:.0f})')

        ax.set_xlabel(f'log10({metric})')
        ax.set_title(f'{name} — {metric}')
        ax.legend(fontsize=8)

plt.show()

## Sweep: where do removed cells stop being special?

For each candidate minimum-count cutoff, take the cells it would remove and ask how many sit in sparse tissue
(distance to the 10th nearest cell in the top 10% for that sample). Enrichment of 1 means removed cells are
spread like any other cells; well above 1 means they concentrate in sparse regions, where you expect low-quality cells.
Orange is the fraction of cells removed.

In [ ]:
CANDIDATE_MIN_COUNTS = np.arange(1, 61)
NEIGHBORS            = 10
SPARSE_FRACTION      = 0.10

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, name in zip(axes.ravel(), SAMPLE_NAMES):
    coordinates = spatial[name]
    neighbor_distance = NearestNeighbors(n_neighbors=NEIGHBORS + 1).fit(coordinates).kneighbors(coordinates)[0][:, -1]
    in_sparse_tissue  = neighbor_distance > np.quantile(neighbor_distance, 1 - SPARSE_FRACTION)

    counts  = qc[name]['total_counts'].to_numpy()
    removed = counts[:, None] < CANDIDATE_MIN_COUNTS[None, :]
    removed_per_candidate = np.maximum(removed.sum(axis=0), 1)
    enrichment = (removed & in_sparse_tissue[:, None]).sum(axis=0) / removed_per_candidate / SPARSE_FRACTION

    ax.plot(CANDIDATE_MIN_COUNTS, enrichment, color='tab:blue')
    ax.axhline(1, color='gray', ls=':')
    ax.set_xlabel('minimum counts')
    ax.set_ylabel('enrichment in sparse tissue (fold)', color='tab:blue')
    ax.set_title(name)

    fraction_axis = ax.twinx()
    fraction_axis.plot(CANDIDATE_MIN_COUNTS, removed.mean(axis=0), color='tab:orange')
    fraction_axis.set_ylabel('fraction of cells removed', color='tab:orange')

    ax.axvline(CURRENT_MIN_COUNTS, color='tab:red', ls='--')
    ax.axvline(cutoffs.loc[name, 'proposed_min_counts'], color='black')

fig.tight_layout()
plt.show()

## Removed cells in space

Fraction of cells removed in each spatial bin. Left: current cutoffs. Right: proposed cutoffs.

In [ ]:
fig, axes = plt.subplots(len(SAMPLE_NAMES), 2, figsize=(12, 5 * len(SAMPLE_NAMES)), squeeze=False)
for row_axes, name in zip(axes, SAMPLE_NAMES):
    counts = qc[name]['total_counts']
    genes  = qc[name]['n_genes']
    removed_by_label = {
        'current':  (counts < CURRENT_MIN_COUNTS) | (genes < CURRENT_MIN_GENES),
        'proposed': (counts < cutoffs.loc[name, 'proposed_min_counts']) | (genes < cutoffs.loc[name, 'proposed_min_genes']),
    }
    for ax, (label, removed) in zip(row_axes, removed_by_label.items()):
        coordinates = spatial[name]
        bins = ax.hexbin(
            coordinates[:, 0], coordinates[:, 1], C=removed.to_numpy().astype(float),
            reduce_C_function=np.mean, gridsize=120, cmap='viridis', vmin=0, vmax=1, rasterized=True,
        )
        fig.colorbar(bins, ax=ax, shrink=0.6)
        ax.set_aspect('equal')
        ax.invert_yaxis()
        ax.set_axis_off()
        ax.set_title(f'{name} — {label}, {removed.mean():.1%} removed')

fig.tight_layout()
plt.show()